In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
from keras.optimizers import Adam
from keras.initializers import RandomNormal
from keras.models import Model
from keras.layers import Input
from keras.layers import Conv2D, Dense, Reshape
from keras.layers import LayerNormalization, Add, MultiHeadAttention, Conv2DTranspose
from keras.layers import LeakyReLU, ReLU, multiply, add
from keras.layers import Layer, Activation
from keras.layers import Concatenate
from keras.layers import Dropout, Embedding
from keras.layers import BatchNormalization , SeparableConv2D
from keras.layers import MaxPooling2D
from tensorflow.keras import layers
from tensorflow.keras.activations import softmax
import tensorflow as tf
from tensorflow.nn import depth_to_space
from tensorflow.image import extract_patches
from numpy import load
from numpy import zeros
from numpy import ones
import numpy as np
from numpy.random import randint

In [ ]:
image_size = 512
patch_size = 32 # Size of the patches to be extract from the input images
num_patches = (image_size // patch_size) ** 2
projection_dim = 256
num_heads = 4
transformer_units = [
    projection_dim * 2,
    projection_dim,
]  # Size of the transformer layers
transformer_layers = 12

In [ ]:
class Patches(layers.Layer):
    def __init__(self, patch_size):
        super(Patches, self).__init__()
        self.patch_size = patch_size
    def get_config(self):
        config = super().get_config().copy()
        config.update({
            'patch_size': self.patch_size,
        })
        return config
    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches


In [ ]:
class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super(PatchEncoder, self).__init__()
        self.num_patches = num_patches
        self.projection = layers.Dense(units=projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, patch):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        encoded = self.projection(patch) + self.position_embedding(positions)
        return encoded

In [ ]:
def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x

In [ ]:
# define the discriminator model
def define_discriminator(image_shape):
   # source image input
    in_src_image = Input(shape=image_shape)
   # target image input
    in_target_image = Input(shape=image_shape)
   # concatenate images channel-wise
    merged = Concatenate()([in_src_image, in_target_image])
    feat =[]
    # Create patches.
    patches = Patches(patch_size)(merged)
    # Encode patches.
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)
    # Create multiple layers of the Transformer block.
    for _ in range(transformer_layers):
        # Layer normalization 1.
        x1 = LayerNormalization(epsilon=1e-6)(encoded_patches)
        # Create a multi-head attention layer.
        attention_output = MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim, dropout=0.1
        )(x1, x1)
        # Skip connection 1.
        x2 = Add()([attention_output, encoded_patches])
        # Layer normalization 2.
        x3 = LayerNormalization(epsilon=1e-6)(x2)
        # MLP.
        x3 = mlp(x3, hidden_units=transformer_units, dropout_rate=0.1)
        # Skip connection 2.
        encoded_patches = Add()([x3, x2])
        feat.append(encoded_patches)
        #print(feat[0].shape)
    feat = Concatenate(axis=-1)([feat[0], feat[1],feat[2],feat[3]])
    # Create a [batch_size, projection_dim] tensor.
    representation = LayerNormalization(epsilon=1e-6)(encoded_patches)
    # Patchgan output
    X_reshape = Reshape((projection_dim, projection_dim,1), name='reshape')(representation)
    d = Conv2D(1, kernel_size=(4,4), strides=(1,1), padding='same',kernel_initializer=RandomNormal(stddev=0.02))(X_reshape)
    patch_out = Activation('sigmoid')(d)
   # define model
    model = Model([in_src_image, in_target_image], patch_out)
   # compile model
    opt = Adam(learning_rate=0.0002, beta_1=0.5)
    model.compile(loss='binary_crossentropy', optimizer=opt, loss_weights=[0.5])
    return model

In [ ]:
def attention_gate(X, g, channel,
                   activation='ReLU',
                   attention='add'):

    activation_func = eval(activation)
    attention_func = eval(attention)

    # mapping the input tensor to the intermediate channel
    theta_att = Conv2D(channel, 1)(X)

    # mapping the gate tensor
    phi_g = Conv2D(channel, 1)(g)

    # ----- attention learning ----- #
    query = attention_func([theta_att, phi_g])

    # nonlinear activation
    f = activation_func()(query)

    # linear transformation
    psi_f = Conv2D(1, 1, use_bias=True)(f)
    # ------------------------------ #

    # sigmoid activation as attention coefficients
    coef_att = Activation('sigmoid')(psi_f)

    # multiplicative attention masking
    X_att = multiply([X, coef_att])

    return X_att

In [ ]:
def attention_unet_base(input_tensor):
  X_skip = []
  init = RandomNormal(stddev=0.02)
  X = Conv2D(64, (4,4), padding='same', activation = 'relu', kernel_initializer=init)(input_tensor)
  X = Conv2D(64, (4,4), strides=(2,2), padding='same', activation = 'relu', kernel_initializer=init)(X)
  X_skip.append(X)

  X = Conv2D(128, (4,4), padding='same', activation = 'relu', kernel_initializer=init)(X)
  X = Conv2D(128, (4,4), strides=(2,2), padding='same', activation = 'relu', kernel_initializer=init)(X)
  X_skip.append(X)

  X = Conv2D(256, (4,4), padding='same', activation = 'relu', kernel_initializer=init)(X)
  X = Conv2D(256, (4,4), strides=(2,2), padding='same', activation = 'relu', kernel_initializer=init)(X)
  X_skip.append(X)

  X = Conv2D(512, (4,4), padding='same', activation = 'relu', kernel_initializer=init)(X)
  X = Conv2D(512, (4,4), strides=(2,2), padding='same', activation = 'relu', kernel_initializer=init)(X)
  X_skip.append(X)


  X_skip = X_skip[::-1]

  X_left = attention_gate(X=X_skip[0], g=X, channel=512, activation='ReLU', attention='add')
  c = Concatenate()([X, X_left])
  X = Conv2DTranspose(512, (4, 4), strides=(2, 2), padding='same', activation='relu', kernel_initializer=init)(c)
  X = Conv2D(512, (4,4), padding='same', activation = 'relu', kernel_initializer=init)(X)

  X_left = attention_gate(X=X_skip[1], g=X, channel=256, activation='ReLU', attention='add')
  c = Concatenate()([X, X_left])
  X = Conv2DTranspose(256, (4, 4), strides=(2, 2), padding='same', activation='relu', kernel_initializer=init)(c)
  X = Conv2D(256, (4,4), padding='same', activation = 'relu', kernel_initializer=init)(X)

  X_left = attention_gate(X=X_skip[2], g=X, channel=128, activation='ReLU', attention='add')
  c = Concatenate()([X, X_left])
  X = Conv2DTranspose(128, (4, 4), strides=(2, 2), padding='same', activation='relu', kernel_initializer=init)(c)
  X = Conv2D(128, (4,4), padding='same', activation = 'relu', kernel_initializer=init)(X)

  X_left = attention_gate(X=X_skip[3], g=X, channel=64, activation='ReLU', attention='add')
  c = Concatenate()([X, X_left])
  X = Conv2DTranspose(64, (4, 4), strides=(2, 2), padding='same', activation='relu', kernel_initializer=init)(c)
  X = Conv2D(64, (4,4), padding='same', activation = 'relu', kernel_initializer=init)(X)


  return X


In [ ]:
# définir le modèle de générateur autonome
def define_generator(input_size, n_labels=3):
    init = RandomNormal(stddev=0.02)
    IN = Input(input_size)
    # Base
    d1 = attention_unet_base(IN)
    # output layer
    X = Conv2D(n_labels, kernel_size=1, padding='same')(d1)
    OUT = Activation('tanh')(X)

    # functional API model
    model = Model(inputs=[IN,], outputs=[OUT,])

    return model

In [ ]:
# define the combined generator and discriminator model, for updating the generator
def define_gan(g_model, d_model, image_shape):
    # make weights in the discriminator not trainable
    d_model.trainable = False
    # define the source image
    in_src = Input(shape=image_shape)
    # connect the source image to the generator input
    gen_out = g_model(in_src)
    # connect the source input and generator output to the discriminator input
    dis_out = d_model([in_src, gen_out])
    # src image as input, generated image and classification output
    model = Model(in_src, [dis_out, gen_out])
    # compile model
    opt = Adam(learning_rate=0.0002, beta_1=0.5)
    #opt = RMSprop(lr=0.00005)
    model.compile(loss=['binary_crossentropy', 'mae'], optimizer=opt, loss_weights=[1,100])
    return model

In [ ]:
# charger et prÃ©parer des images de formation
def load_real_samples(filename):
	# charger des tableaux compressÃ©s
	data = load(filename)
	# dÃ©compresser arrays
	X1, X2 = data['arr_0'], data['arr_1']
	# echelle de [0,255] Ã  [-1,1]
	X1 = (X1 - 127.5) / 127.5
	X2 = (X2 - 127.5) / 127.5
	return [X1, X2]

In [ ]:
# select a batch of random samples, returns images and target
def generate_real_samples(dataset, n_samples, patch_shape):
    # unpack dataset
    trainA, trainB = dataset
    # choose random instances
    ix = randint(0, trainA.shape[0], n_samples)
    # retrieve selected images
    X1, X2 = trainA[ix], trainB[ix]
    # generate ✬real✬ class labels (1)
    y = ones((n_samples, patch_shape, patch_shape, 1))
    return [X1, X2], y


In [ ]:
# generate a batch of images, returns images and targets
def generate_fake_samples(g_model, samples, patch_shape):
    # generate fake instance
    X = g_model.predict(samples)
    # create ✬fake✬ class labels (0)
    y = zeros((len(X), patch_shape, patch_shape, 1))
    return X, y


In [ ]:
# train pix2pix models
def train(d_model, g_model, gan_model, dataset, n_epochs=200, n_batch=1):
    # unpack dataset
    trainA, trainB = dataset
    # dÃ©terminer la forme carrÃ©e de sortie du discriminateur
    n_patch = d_model.output_shape[1]
    # calculate the number of batches per training epoch
    bat_per_epo = int(len(trainA) / n_batch)
    # calculate the number of training iterations
    n_steps = bat_per_epo * n_epochs
    # manually enumerate epochs
    for i in range(n_steps):
      # select a batch of real samples
      [X_realA, X_realB], y_real = generate_real_samples(dataset, n_batch, n_patch)
      # generate a batch of fake samples
      X_fakeB, y_fake = generate_fake_samples(g_model, X_realA, n_patch)
      # update discriminator for real samples
      d_loss1 = d_model.train_on_batch([X_realA, X_realB], y_real)
      # update discriminator for generated samples
      d_loss2 = d_model.train_on_batch([X_realA, X_fakeB], y_fake)
      # update the generator
      g_loss, _, _ = gan_model.train_on_batch(X_realA, [y_real, X_realB])
      # summarize performance
      print('>%d, d1[%.3f] d2[%.3f] g[%.3f]' % (i+1, d_loss1, d_loss2, g_loss))


    # sauvgarder le modele generateur
    filename2 = '/content/gdrive/MyDrive/hair_removal/EAGAN_%06d.h5' % (i+1)
    g_model.save(filename2)
    print('>Saved: %s' % (filename2))

In [ ]:
# charger les donnÃ©es de l'image
dataset = load_real_samples('/content/gdrive/MyDrive/data_hair_removal800_512.npz')
# dÃ©finir une forme d'entrÃ©e basÃ©e sur l'ensemble de donnÃ©es chargÃ©
image_shape = (512, 512,3)
# dÃ©finir les modÃ¨les
d_model = define_discriminator(image_shape)
g_model = define_generator(image_shape)
# dÃ©finir le modÃ¨le composite
gan_model = define_gan(g_model, d_model, image_shape)
# entrainer le modÃ¨le
train(d_model, g_model, gan_model, dataset)